In [ ]:
# !nvidia-smi

In [ ]:
# import torch
# print(torch.cuda.is_available())

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton
!pip install mlflow rouge-score bert-score datasets pandas scikit-learn
!pip install langdetect
!pip install pydantic

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-r3yl1dh4/unsloth_56444fcc34ed419aaf99323bd30dc16f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-r3yl1dh4/unsloth_56444fcc34ed419aaf99323bd30dc16f
  Resolved https://github.com/unslothai/unsloth.git to commit 7d0d2f256c27dffd5437b20595adbe66f9d6272e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset
from pydantic import BaseModel, field_validator
from typing import List
import openai
import json
import time
from langdetect import detect, LangDetectException
from sklearn.model_selection import train_test_split
from collections import Counter
import random
from google.colab import drive

In [ ]:
# Loading dataset

dataset = load_dataset("amazon_polarity", split="train")
df = pd.DataFrame(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
df = pd.DataFrame(dataset)

print(f"Total reviews: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

Total reviews: 3,600,000
Columns: ['label', 'title', 'content']

Label distribution:
label
1    1800000
0    1800000
Name: count, dtype: int64


In [ ]:
df = df.rename(columns={"content": "text", "label": "sentiment"})

# Filtering the reviews - keeping the reviews with meaningful length
df = df[df['text'].str.len().between(100, 2000)].copy()

# Dropping duplicates
df = df.drop_duplicates(subset=['text'])

print(f"After cleaning: {len(df):,} reviews")

After cleaning: 3,518,237 reviews


In [ ]:
# Language detection on a sample first to check the ratio
# (Running on ALL 3.6M rows would take hours — we don't need that)
# Strategy: sample MORE than we need, filter, then take 3000

sample_size = 20000  # Grab 10K — enough buffer after filtering
pos_sample = df[df['sentiment'] == 1].sample(n=sample_size, random_state=42)
neg_sample = df[df['sentiment'] == 0].sample(n=sample_size, random_state=42)
sample_df = pd.concat([pos_sample, neg_sample]).reset_index(drop=True)

print(f"Sampled {len(sample_df)} reviews for language detection...")

Sampled 40000 reviews for language detection...


In [ ]:
# Detect language
def safe_detect(text):
    try:
        return detect(text)
    except:
        return 'unknown'

sample_df['lang'] = sample_df['text'].apply(safe_detect)

print(f"\nLanguage distribution:")
print(sample_df['lang'].value_counts().head(10))

# Keep only English
english_df = sample_df[sample_df['lang'] == 'en'].copy()
print(f"\nEnglish reviews: {len(english_df):,}")


Language distribution:
lang
en    39886
es       86
fr       10
pt        6
de        4
it        3
hr        1
tl        1
nl        1
cy        1
Name: count, dtype: int64

English reviews: 39,886


In [ ]:
# Creating a balanced sample dataset - 1500 positive, 1500 negative

pos = english_df[english_df['sentiment'] == 1].sample(
    n=min(1500, len(english_df[english_df['sentiment'] == 1])),
    random_state=42
)
neg = english_df[english_df['sentiment'] == 0].sample(
    n=min(1500, len(english_df[english_df['sentiment'] == 0])),
    random_state=42
)

balanced_df = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

# Drop the lang column - no use of it now
balanced_df = balanced_df.drop(columns=['lang'])

print(f"Final balanced dataset: {len(balanced_df)} English reviews")
print(f"Positive: {(balanced_df['sentiment']==1).sum()}")
print(f"Negative: {(balanced_df['sentiment']==0).sum()}")

Final balanced dataset: 3000 English reviews
Positive: 1500
Negative: 1500


In [ ]:
# Look into one of each category reviews
for sent in [0, 1]:
  sample = balanced_df[balanced_df['sentiment'] == sent].iloc[0]
  label = "NEGATIVE" if sent == 0 else "POSITIVE"
  print(f"\n{'='*50}")
  print(f"[{label}] {sample['text'][:300]}...")


[NEGATIVE] 1. Hacksaw Jim Duggan beats people up.2. The Nasty Boys are just plain nasty.3. Bret Hart is the best there is, the best there was and the best there ever will be.4. The Man in Black has got a tombstone just for you.5. The Undertaker is the most powerful entity in the World Wrestling Federation.6. "...

[POSITIVE] Gilead's blood was the first black library book I read and I absolutely loved it. The emotion and character relation between the characters is outstanding. If there was anything I didn't like about the book it was the dreaded short-story format black library is famous for writing....


1. Added up validation - validating the data types before serialization - using pydantic
2. Ratings were decimals numbers - So improved the prompt


In [ ]:
# # Generating structured labels - with GPT model

# client = openai.OpenAI(api_key="")

# LABELING_PROMT = """Analyze this product review and extract a structured summary.

# Review: {review_text}
# Sentiment: {sentiment}

# Respond in EXACTLY this json format and nothing else:
# {{
#   "pros": ["pro1","pro2"],
#   "cons": ["con1","con2"],
#   "verdict": "One sentence overall verdict",
#   "rating": N
# }}

# Rules:
# - Extract 1-4 pros and 1-4 cons from the ACTUAL review content
# - Each pro/con should be a concise phrase (5-15 words)
# - The verdict should be one clear sentence
# - Assign a rating that is exactly one of: 1, 2, 3, 4, or 5 (integers only, no decimals): 1=terrible, 2=bad, 3=average/mixed, 4=good, 5=excellent
# - If the review has NO positives, use: ["No notable strengths mentioned"]
# - If the review has NO negatives, use: ["No significant drawbacks mentioned"]
# - Even positive reviews might have cons, and negative ones might have pros
# - BE SPECIFIC - PULL ACTUAL DETAILS FROM THE REVIEW, NOT GENERIC PHRASES"""

# def generate_label(review_text, sentiment, max_chars=800, max_retries=3):
#   sentiment_str = "POSITIVE" if sentiment == 1 else "NEGATIVE"
#   for attempt in range (max_retries):
#     try:
#       # making API call
#       response = client.chat.completions.create(
#           model="gpt-4o-mini",
#           messages=[{
#               "role": "user",
#               "content": LABELING_PROMT.format(
#                   review_text = review_text[:max_chars],
#                   sentiment = sentiment_str
#               )
#           }],
#           temperature=0.3,
#           max_tokens=300,
#           response_format={"type": "json_object"} # JSON mode
#       )
#       result = json.loads(response.choices[0].message.content)

#       # ========= VALIDATION + CLEANING ====================

#       # Check required keys
#       if not all(k in result for k in ["pros", "cons", "verdict"]):
#         print(f"Missing keys, retrying... Got: {list(result.keys())}")
#         continue

#       # Force pros and cons to be lists of strings
#       result["pros"] = [str(p) for p in list(result.get("pros", []))]
#       result["cons"] = [str(c) for c in list(result.get("cons", []))]

#       # Force verdict to be a string
#       result["verdict"] = str(result.get("verdict", "Mixed experience."))

#       # Force rating to be an integer 1-5
#       rating = result.get("rating", 3)
#       result["rating"] = max(1, min(5, int(5, int(round(float(rating))))))

#       # Validate pros/cons aren't empty
#       if len(result["pros"]) == 0:
#         result["pros"] = ["No notable strengths mentioned"]
#       if len(result["cons"]) == 0:
#         result["cons"] = ["No notable drawbacks mentioned"]

#       # ========= VALIDATION END ==============================

#         return result

#     except Exception as e:
#       if attempt < max_retries - 1:
#         time.sleep(2)
#       else:
#         print(f"Failed: {e}")
#   return None

In [ ]:
# Define the data schema with Pydantic
class ReviewLabel(BaseModel):
    """Schema for a structured product review summary.

    Pydantic does three things:
    1. Type checking — ensures pros is a list, rating is an int, etc.
    2. Validation — runs custom rules (rating must be 1-5, lists can't be empty)
    3. Coercion — automatically converts compatible types (3.5 → 4, set → list)
    """

    pros: List[str]
    cons: List[str]
    verdict: str
    rating: int

    @field_validator('rating', mode='before')
    @classmethod
    def clamp_rating_1_to_5(cls, v):
        """
        mode='before' means this runs BEFORE Pydantic tries to convert to int.
        So if GPT returns 3.5 (a float), we catch it here first.

        Flow: 3.5 → round → 4 → clamp between 1-5 → 4
        Flow: 0   → round → 0 → clamp → 1
        Flow: 7   → round → 7 → clamp → 5
        """
        return max(1, min(5, int(round(float(v)))))

    @field_validator('pros', 'cons', mode='before')
    @classmethod
    def ensure_non_empty_string_list(cls, v):
        """
        mode='before' means this runs BEFORE Pydantic checks List[str].
        Handles multiple bad formats GPT might return:
        - A set instead of a list → convert to list
        - An empty list → fill with default
        - A list with non-strings → convert each to string
        - A single string instead of a list → wrap in list
        """
        # If GPT returned a single string instead of a list
        if isinstance(v, str):
            v = [v]

        # If it's a set, tuple, or other iterable → make it a list
        if not isinstance(v, list):
            v = list(v)

        # Convert every item to string (handles ints, None, etc.)
        v = [str(item) for item in v if item is not None]

        # Don't allow empty lists
        if len(v) == 0:
            # We'll figure out if it's pros or cons from the field name
            # For now return a generic default — the field_validator
            # doesn't easily know its own field name, so we handle both
            v = ["No notable points mentioned"]

        return v

    @field_validator('verdict', mode='before')
    @classmethod
    def ensure_verdict_is_string(cls, v):
        """Handle cases where verdict might be None or a non-string"""
        if v is None or v == "":
            return "Mixed experience overall."
        return str(v)

In [ ]:
# Generating structured labels - with GPT model

# Production generate_label with Pydantic validation
import openai
import json
import time

client = openai.OpenAI(api_key="")

LABELING_PROMPT = """Analyze this product review and extract a structured summary.

Review: {review_text}
Sentiment: {sentiment}

Respond in EXACTLY this JSON format and nothing else:
{{
  "pros": ["pro1", "pro2"],
  "cons": ["con1", "con2"],
  "verdict": "One sentence overall verdict",
  "rating": N
}}

Rules:
- Extract 1-4 pros and 1-4 cons from the ACTUAL review content
- Each pro/con should be a concise phrase (5-15 words)
- The verdict should be one clear sentence
- Assign a rating that is exactly one of: 1, 2, 3, 4, or 5 (integers only, no decimals): 1=terrible, 2=bad, 3=average/mixed, 4=good, 5=excellent
- If the review has NO positives, use: ["No notable strengths mentioned"]
- If the review has NO negatives, use: ["No significant drawbacks mentioned"]
- Even positive reviews might have cons, and negative ones might have pros
- BE SPECIFIC - PULL ACTUAL DETAILS FROM THE REVIEW, NOT GENERIC PHRASES"""

def generate_label(review_text, sentiment, max_chars=800, max_retries=3):
    sentiment_str = "positive" if sentiment == 1 else "negative"

    for attempt in range(max_retries):
        try:
            # Step 1: Call the API
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{
                    "role": "user",
                    "content": LABELING_PROMPT.format(
                        review_text=review_text[:max_chars],
                        sentiment=sentiment_str
                    )
                }],
                temperature=0.3,
                max_tokens=300,
                response_format={"type": "json_object"}
            )

            # Step 2: Parse JSON string → Python dict
            raw_result = json.loads(response.choices[0].message.content)

            # Step 3: Validate + clean with Pydantic
            # This one line handles ALL the issues:
            #   - 3.5 rating → rounds to 4
            #   - set → list
            #   - empty pros → default text
            #   - missing verdict → default text
            #   - rating=7 → clamped to 5
            label = ReviewLabel(**raw_result)

            # Step 4: Convert back to a plain dict for storage
            return label.model_dump()

        except json.JSONDecodeError:
            # API returned something that isn't valid JSON
            print(f"  Attempt {attempt+1}: Invalid JSON, retrying...")
            time.sleep(2)

        except Exception as e:
            # Pydantic validation failed, or API error, or network issue
            if attempt < max_retries - 1:
                print(f"  Attempt {attempt+1}: {type(e).__name__}: {e}, retrying...")
                time.sleep(2)
            else:
                print(f"  FAILED after {max_retries} attempts: {e}")

    return None

In [ ]:
# # To decide on the truncation length - max_chars - lets pull 20 longest reviews - to get the understanding

# # longest_reviews = balanced_df.sort_values(by='text', key=lambda x: x.str.len(), ascending=False)[:20]
# longest_reviews = balanced_df[balanced_df['text'].str.len() > 800].sample(20, random_state=42)

# for _, row in longest_reviews.head(5).iterrows():
#   print(f"\nFull review ({len(row['text'])} chars:)")
#   print(row['text'][:200] + "...")

#   label_500 = generate_label(row['text'], row['sentiment'], max_chars=500)
#   label_800 = generate_label(row['text'], row['sentiment'], max_chars=800)
#   label_1000 = generate_label(row['text'], row['sentiment'], max_chars=1000)
#   label_full = generate_label(row['text'], row['sentiment'], max_chars=5000)

#   print(f"@500: {label_500['pros']}")
#   print(f"@800: {label_800['pros']}")
#   print(f"@1000: {label_1000['pros']}")
# #   print(f"@full: {label_full['pros']}")




In [ ]:
# Decision log
# Truncation test on 20 long reviews (>800 chars):
# - @500 vs @800: reviews had richer labels at 800
# - @800 vs @full: not much difference in label quality
# - Decision: truncate at 800 chars
# - Cost savings: fewer input tokens vs no truncation

In [ ]:
# Processing labels for the reviews - based on above results - max_chars @800 and @full produce identical results in every case. @500 misses some detail. So 800 is the right cutoff — we get almost the same information without paying for extra tokens.

labels = []
failed = 0

for i, (_, row) in enumerate(balanced_df.iterrows()):
    if i % 100 == 0:
        print(f"Labeling... {i}/{len(balanced_df)} ({failed} failed so far)")

    label = generate_label(row['text'], row['sentiment'])
    if label:
        labels.append(label)
    else:
        labels.append(None)
        failed += 1

balanced_df['label'] = labels
balanced_df = balanced_df.dropna(subset=['label'])
print(f"\nDone! {len(balanced_df)} labeled, {failed} failed")

# Extract rating - now guaranteed to be a clean int 1-5
balanced_df['rating'] = balanced_df['label'].apply(lambda x: x['rating'])
print(f"\nRating distribution:")
print(balanced_df['rating'].value_counts().sort_index())
# No more 3.5 or 4.5 - Pydantic fixed them

Labeling... 0/3000 (0 failed so far)
Labeling... 100/3000 (0 failed so far)
Labeling... 200/3000 (0 failed so far)
Labeling... 300/3000 (0 failed so far)
Labeling... 400/3000 (0 failed so far)
Labeling... 500/3000 (0 failed so far)
Labeling... 600/3000 (0 failed so far)
Labeling... 700/3000 (0 failed so far)
Labeling... 800/3000 (0 failed so far)
Labeling... 900/3000 (0 failed so far)
Labeling... 1000/3000 (0 failed so far)
Labeling... 1100/3000 (0 failed so far)
Labeling... 1200/3000 (0 failed so far)
Labeling... 1300/3000 (0 failed so far)
Labeling... 1400/3000 (0 failed so far)
Labeling... 1500/3000 (0 failed so far)
Labeling... 1600/3000 (0 failed so far)
Labeling... 1700/3000 (0 failed so far)
Labeling... 1800/3000 (0 failed so far)
Labeling... 1900/3000 (0 failed so far)
Labeling... 2000/3000 (0 failed so far)
Labeling... 2100/3000 (0 failed so far)
Labeling... 2200/3000 (0 failed so far)
Labeling... 2300/3000 (0 failed so far)
Labeling... 2400/3000 (0 failed so far)
Labeling... 

Now it's observed that the rating distribution is imbalanced. - "DATA IMBALANCE PROBLEM"

1. Rating '2' and rating '5' reviews are way too many, while rating '3' reviews are too few
2. When the model is directly trained on this, it will see rating '2' reviews 21x more often than rating '3' reviews. Which means the model gets good at summarizing negative reviews and barely learns how to handle mixed/ neutral ones.

Looking at strategies for imbalanced data:

1. Down sampling: This can't be done because, cutting (down sizing) all the groups to match rating '3' size will eliminate 90% of the data.
2. Stratified sampling: This can't be directly applied because, the percentage of rating '3' will still remain low. Isn't solving our problem.
3. Cap sampling: Cap the large groups and keep the small group as is.

So let's first Cap the data and apply stratified sampling. On this then apply over sampling to minority in the training set only!!

Capping overrepresented groups to 450 reviews


In [ ]:
MAX_PER_RATING = 450
capped_groups = []
for rating in sorted(balanced_df['rating'].unique()):
  group = balanced_df[balanced_df['rating'] == rating]
  if len(group) > MAX_PER_RATING:
    group = group.sample(n=MAX_PER_RATING, random_state=42)
  capped_groups.append(group)

balanced_df = pd.concat(capped_groups).sample(frac=1, random_state=42).reset_index(drop=True)
print("\nAfter capping + augmenting:")
print(balanced_df['rating'].value_counts().sort_index())


After capping + augmenting:
rating
1    450
2    450
3     62
4    450
5    450
Name: count, dtype: int64


In [ ]:
# Preparing training data - in ALPACA format - three-part structure that instruction-tuned models expect: an instruction, an input, and an output

def format_training_example(row):
  # A system level instruction
  instruction = (
      "You are a product review analyser. Given a customer review, generate a structured summary with pros, cons, and an overall verdict."
  )

  input_text = f"Review: {row['text']} \nRating:{row['rating']}/5"

  pros = "\n".join(f"- {p}" for p in row['label']['pros'])
  cons = "\n".join(f"- {c}" for c in row['label']['cons'])

  output_text = f"**Pros:**\n{pros}\n\n**Cons:**\n{cons}\n\n**Verdict:** {row['label']['verdict']}"

  return {
      "instruction": instruction,
      "input": input_text,
      "output": output_text
  }


In [ ]:
# Formatting examples
training_data = [format_training_example(row) for _,row in balanced_df.iterrows()]
ratings = balanced_df['rating'].tolist()

In [ ]:
# Stratified sampling - stratified split
train_data, temp_data, train_ratings, temp_ratings = train_test_split(training_data, ratings, test_size=0.2, random_state=42, stratify=ratings)
val_data, test_data, val_ratings, test_ratings = train_test_split(temp_data, temp_ratings, test_size=0.5, random_state=42, stratify=temp_ratings)

In [ ]:
def oversample_minority(train_examples, train_ratings, target_per_class=None):
    """Duplicate minority class examples so the model sees them more often"""

    counts = Counter(train_ratings)
    if target_per_class is None:
        target_per_class = max(counts.values())

    # Start with all existing examples
    oversampled = list(train_examples)

    # Adding duplicates for underrepresented ratings
    for rating, count in counts.items():
        if count < target_per_class:
            deficit = target_per_class - count
            # Get all examples of this rating
            class_examples = [
                ex for ex, r in zip(train_examples, train_ratings) if r == rating
            ]
            # Randomly duplicate to fill the deficit
            random.seed(42)
            extras = random.choices(class_examples, k=deficit)
            oversampled.extend(extras)
            print(f"  Rating {rating}: {count} → {count + deficit} (+{deficit} duplicates)")

    random.shuffle(oversampled)
    return oversampled

In [ ]:
# Oversampling minority in Training set ONLY

print("\nOversampling training set:")
train_data = oversample_minority(train_data, train_ratings, target_per_class=360)


Oversampling training set:
  Rating 3: 49 → 360 (+311 duplicates)


In [ ]:
# Verify
print(f"\nFinal sizes:")
print(f"  Train: {len(train_data)} (oversampled)")
print(f"  Val:   {len(val_data)} (real distribution)")
print(f"  Test:  {len(test_data)} (real distribution)")


Final sizes:
  Train: 1800 (oversampled)
  Val:   186 (real distribution)
  Test:  187 (real distribution)


In [ ]:
# Wrapping in MISTRAL chat template
# <s>     ----> Start of sequence token
# [INST]  ----> Everything after this is from the user
# [/INST] ----> User message ends here. Model should respond now
# </s>    ----> End of sequence

PROMT_TEMPLATE = """<s>[INST] {instruction}

{input} [/INST]
{output}</s>"""

# Supervised Fine Tuning - SFT
def format_for_sft(example):
  return {"text": PROMT_TEMPLATE.format(**example)}


train_dataset = Dataset.from_list([format_for_sft(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_for_sft(ex) for ex in val_data])

print(f"Train data: {len(train_dataset)}")
print(f"Evaluation dataset: {len(val_dataset)}")

print(f"\nSample:\n{train_dataset[0]['text'][:500]}...")

Train data: 1800
Evaluation dataset: 186

Sample:
<s>[INST] You are a product review analyser. Given a customer review, generate a structured summary with pros, cons, and an overall verdict.

Review: This is a really good look at the Hulk and Zeus! I have been waiting years for this to come to DVD and now it has. I got the VHS when it first came out and still have it. I am a wrestling buff, starting with the WWF and have watched the Hulk's career. I even have some magazines from the 60's with a young Hulk so this movie was for me. 2 giants in t...


In [ ]:
# Step 1: Find which split has the problem
for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    try:
        json.dumps(data)
        print(f"{name}: OK")
    except TypeError as e:
        print(f"{name}: BROKEN — {e}")

train: OK
val: OK
test: OK


In [ ]:
# Save everything to Google Drive

save_dir = '/content/drive/MyDrive/review-summarizer'
!mkdir -p {save_dir}

# Raw splits (for evaluation later)
for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    with open(f"{save_dir}/{name}_data.json", "w") as f:
        json.dump(data, f)
    print(f"Saved {name}_data.json ({len(data)} examples)")

# SFT-formatted datasets (for training)
train_dataset.to_json(f"{save_dir}/train_dataset_sft.jsonl")
val_dataset.to_json(f"{save_dir}/val_dataset_sft.jsonl")
print(f"Saved SFT datasets")

# Full DataFrame as backup
balanced_df.to_parquet(f"{save_dir}/balanced_reviews.parquet")
print(f"Saved balanced_reviews.parquet")

print(f"\nAll files:")
!ls -lh {save_dir}

Saved train_data.json (1800 examples)
Saved val_data.json (186 examples)
Saved test_data.json (187 examples)


Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved SFT datasets
Saved balanced_reviews.parquet

All files:
total 4.2M
-rw------- 1 root root 775K Apr 18 07:25 balanced_reviews.parquet
-rw------- 1 root root 158K Apr 18 07:25 test_data.json
-rw------- 1 root root 1.5M Apr 18 07:25 train_data.json
-rw------- 1 root root 1.5M Apr 18 07:25 train_dataset_sft.jsonl
-rw------- 1 root root 156K Apr 18 07:25 val_data.json
-rw------- 1 root root 155K Apr 18 07:25 val_dataset_sft.jsonl
